# Dendritron Plasticity Limit Sweep v0.8

Single-cell, process-isolated 42-run stress test.

In [ ]:
from pathlib import Path
import subprocess, sys
WORKDIR = Path('/content') if Path('/content').exists() else Path('/mnt/data')
WORKDIR.mkdir(parents=True, exist_ok=True)
code07 = 'from __future__ import annotations\n\nimport json\nimport math\nimport time\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Tuple\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.cluster import KMeans\nfrom sklearn.neural_network import MLPClassifier\n\nOUT = Path(\'/mnt/data\')\nSEED = 42\n\n\ndef softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:\n    z = x - np.max(x, axis=axis, keepdims=True)\n    e = np.exp(z)\n    return e / np.sum(e, axis=axis, keepdims=True)\n\n\n@dataclass\nclass Branch:\n    branch_id: int\n    class_id: int\n    center: np.ndarray\n    sigma: float\n    count: int\n    created_step: int\n    last_seen: int\n    active: bool = True\n    archived: bool = False\n    damage_disabled: bool = False\n    distance_ema: float = 0.0\n    error_ema: float = 0.0\n    utility_ema: float = 0.0\n    buffer: List[np.ndarray] = field(default_factory=list)\n\n    def score(self, x: np.ndarray) -> float:\n        if not self.active or self.damage_disabled:\n            return 0.0\n        d2 = float(np.sum((x - self.center) ** 2))\n        s2 = max(self.sigma * self.sigma, 1e-8)\n        return math.exp(-0.5 * d2 / s2)\n\n\nclass PlasticDendritronWeb:\n    """Online prototype tissue with explicit ownership and structural plasticity.\n\n    Each class is a functional region. Each region owns local RBF branches.\n    Structural edits are local and logged: grow, split, merge, retire, reactivate,\n    damage, repair. Candidate branches are quarantined against certificates from\n    existing regions before activation.\n    """\n\n    def __init__(\n        self,\n        input_dim: int,\n        base_sigma: float = 1.50,\n        min_sigma: float = 0.35,\n        max_sigma: float = 2.8,\n        grow_z: float = 2.05,\n        wrong_score_trigger: float = 0.48,\n        center_lr_cap: float = 0.08,\n        sigma_lr: float = 0.03,\n        certificate_size: int = 72,\n        quarantine_margin: float = 0.02,\n        retirement_steps: int = 1100,\n        split_interval: int = 350,\n        merge_interval: int = 500,\n        retire_interval: int = 250,\n        buffer_size: int = 80,\n        rng: Optional[np.random.Generator] = None,\n    ):\n        self.input_dim = input_dim\n        self.base_sigma = base_sigma\n        self.min_sigma = min_sigma\n        self.max_sigma = max_sigma\n        self.grow_z = grow_z\n        self.wrong_score_trigger = wrong_score_trigger\n        self.center_lr_cap = center_lr_cap\n        self.sigma_lr = sigma_lr\n        self.certificate_size = certificate_size\n        self.quarantine_margin = quarantine_margin\n        self.retirement_steps = retirement_steps\n        self.split_interval = split_interval\n        self.merge_interval = merge_interval\n        self.retire_interval = retire_interval\n        self.buffer_size = buffer_size\n        self.rng = rng or np.random.default_rng(0)\n\n        self.regions: Dict[int, List[Branch]] = {}\n        self.archives: Dict[int, List[Branch]] = {}\n        self.certificates: Dict[int, List[np.ndarray]] = {}\n        self.branch_counter = 0\n        self.step = 0\n        self.events: List[dict] = []\n        self.last_recovery_event: Dict[int, int] = {}\n\n    # ---------- inference ----------\n    def class_score(self, x: np.ndarray, class_id: int) -> float:\n        branches = self.regions.get(class_id, [])\n        if not branches:\n            return 0.0\n        scores = [b.score(x) for b in branches]\n        # Noisy-OR coalition: one strong local branch can recognize a mode,\n        # while multiple moderate branches can cooperate.\n        return 1.0 - float(np.prod([1.0 - min(max(s, 0.0), 0.999999) for s in scores]))\n\n    def class_score_batch(self, X: np.ndarray, class_id: int) -> np.ndarray:\n        branches = [b for b in self.regions.get(class_id, []) if b.active and not b.damage_disabled]\n        if not branches:\n            return np.zeros(len(X), dtype=float)\n        centers = np.stack([b.center for b in branches])\n        sigmas = np.array([max(b.sigma, self.min_sigma) for b in branches])\n        d2 = np.sum((X[:, None, :] - centers[None, :, :]) ** 2, axis=2)\n        scores = np.exp(-0.5 * d2 / (sigmas[None, :] ** 2))\n        scores = np.clip(scores, 0.0, 0.999999)\n        return 1.0 - np.prod(1.0 - scores, axis=1)\n\n    def score_vector(self, x: np.ndarray, classes: Optional[List[int]] = None) -> Tuple[np.ndarray, List[int]]:\n        cls = sorted(self.regions) if classes is None else list(classes)\n        if not cls:\n            return np.empty(0), []\n        return np.array([self.class_score(x, c) for c in cls], dtype=float), cls\n\n    def predict_one(self, x: np.ndarray) -> int:\n        scores, cls = self.score_vector(x)\n        if len(cls) == 0:\n            return -1\n        return int(cls[int(np.argmax(scores))])\n\n    def predict(self, X: np.ndarray) -> np.ndarray:\n        cls = sorted(self.regions)\n        if not cls:\n            return np.full(len(X), -1, dtype=int)\n        scores = np.stack([self.class_score_batch(X, c) for c in cls], axis=1)\n        return np.array(cls, dtype=int)[np.argmax(scores, axis=1)]\n\n    # ---------- certificates ----------\n    def _add_certificate(self, x: np.ndarray, y: int) -> None:\n        certs = self.certificates.setdefault(y, [])\n        if len(certs) < self.certificate_size:\n            certs.append(x.copy())\n        else:\n            # Reservoir-like replacement biased toward recent evidence.\n            if self.rng.random() < 0.08:\n                certs[int(self.rng.integers(0, len(certs)))] = x.copy()\n\n    def certificate_accuracy(self, class_id: int) -> float:\n        certs = self.certificates.get(class_id, [])\n        if not certs:\n            return float(\'nan\')\n        X = np.stack(certs)\n        return float(np.mean(self.predict(X) == class_id))\n\n    # ---------- branch lifecycle ----------\n    def _new_branch(self, class_id: int, center: np.ndarray, sigma: float, reason: str) -> Branch:\n        self.branch_counter += 1\n        b = Branch(\n            branch_id=self.branch_counter,\n            class_id=class_id,\n            center=center.copy(),\n            sigma=float(np.clip(sigma, self.min_sigma, self.max_sigma)),\n            count=1,\n            created_step=self.step,\n            last_seen=self.step,\n        )\n        self.events.append({\n            \'step\': self.step,\n            \'event\': \'grow\',\n            \'class_id\': class_id,\n            \'branch_id\': b.branch_id,\n            \'reason\': reason,\n            \'sigma\': b.sigma,\n        })\n        return b\n\n    def _candidate_sigma(self, x: np.ndarray, class_id: int) -> float:\n        sigma = self.base_sigma\n        others = [np.stack(pts) for c, pts in self.certificates.items() if c != class_id and pts]\n        if others:\n            O = np.vstack(others)\n            d = float(np.sqrt(np.min(np.sum((O - x[None, :]) ** 2, axis=1))))\n            sigma = min(sigma, max(self.min_sigma, 0.42 * d))\n        return float(np.clip(sigma, self.min_sigma, self.max_sigma))\n\n    def _quarantine_safe(self, candidate: Branch) -> bool:\n        # Vectorized bounded ownership-certificate check.\n        for true_c, pts in self.certificates.items():\n            if true_c == candidate.class_id or not pts:\n                continue\n            X = np.stack(pts)\n            incumbent = self.class_score_batch(X, true_c)\n            d2 = np.sum((X - candidate.center[None, :]) ** 2, axis=1)\n            cand = np.exp(-0.5 * d2 / max(candidate.sigma ** 2, 1e-8))\n            if np.any(cand > incumbent + self.quarantine_margin):\n                return False\n        return True\n\n    def _propose_branch(self, x: np.ndarray, class_id: int, reason: str) -> Optional[Branch]:\n        # Prefer reactivation of dormant local structure.\n        archived = self.archives.get(class_id, [])\n        if archived:\n            distances = [np.linalg.norm(x - b.center) / max(b.sigma, self.min_sigma) for b in archived]\n            idx = int(np.argmin(distances))\n            if distances[idx] <= 2.0:\n                b = archived.pop(idx)\n                b.active = True\n                b.archived = False\n                b.damage_disabled = False\n                b.last_seen = self.step\n                self.regions.setdefault(class_id, []).append(b)\n                self.events.append({\n                    \'step\': self.step,\n                    \'event\': \'reactivate\',\n                    \'class_id\': class_id,\n                    \'branch_id\': b.branch_id,\n                    \'reason\': reason,\n                    \'normalized_distance\': float(distances[idx]),\n                })\n                return b\n\n        sigma = self._candidate_sigma(x, class_id)\n        # Shrink until the candidate passes protected certificates.\n        for _ in range(8):\n            b = self._new_branch(class_id, x, sigma, reason)\n            if self._quarantine_safe(b):\n                self.regions.setdefault(class_id, []).append(b)\n                return b\n            # The logged grow attempt becomes a quarantine rejection.\n            self.events[-1][\'event\'] = \'reject\'\n            self.events[-1][\'reason\'] = f\'{reason}:certificate_conflict\'\n            sigma *= 0.72\n            if sigma < self.min_sigma:\n                break\n        return None\n\n    def _update_branch(self, b: Branch, x: np.ndarray, prediction_was_correct: bool) -> None:\n        dist = float(np.linalg.norm(x - b.center))\n        b.count += 1\n        lr = min(self.center_lr_cap, 1.0 / math.sqrt(b.count))\n        b.center = (1.0 - lr) * b.center + lr * x\n        b.distance_ema = 0.97 * b.distance_ema + 0.03 * dist\n        target_sigma = np.clip(max(self.base_sigma, 1.35 * b.distance_ema), self.min_sigma, self.max_sigma)\n        b.sigma = float((1.0 - self.sigma_lr) * b.sigma + self.sigma_lr * target_sigma)\n        b.error_ema = 0.97 * b.error_ema + 0.03 * (0.0 if prediction_was_correct else 1.0)\n        b.utility_ema = 0.995 * b.utility_ema + 0.005\n        b.last_seen = self.step\n        b.buffer.append(x.copy())\n        if len(b.buffer) > self.buffer_size:\n            b.buffer.pop(0)\n\n    def _split_candidates(self) -> None:\n        for class_id, branches in list(self.regions.items()):\n            for b in list(branches):\n                if not b.active or b.damage_disabled or len(b.buffer) < 48:\n                    continue\n                X = np.stack(b.buffer)\n                total_sse = float(np.sum((X - X.mean(axis=0)) ** 2))\n                if total_sse <= 1e-8:\n                    continue\n                # Lightweight deterministic 2-means; avoids global optimizer overhead.\n                X0 = X - X.mean(axis=0, keepdims=True)\n                _, _, vt = np.linalg.svd(X0, full_matrices=False)\n                axis = vt[0]\n                proj = X0 @ axis\n                c0 = X[int(np.argmin(proj))].copy()\n                c1 = X[int(np.argmax(proj))].copy()\n                labels = np.zeros(len(X), dtype=int)\n                for _ in range(8):\n                    d0 = np.sum((X - c0) ** 2, axis=1)\n                    d1 = np.sum((X - c1) ** 2, axis=1)\n                    labels = (d1 < d0).astype(int)\n                    if np.any(labels == 0): c0 = X[labels == 0].mean(axis=0)\n                    if np.any(labels == 1): c1 = X[labels == 1].mean(axis=0)\n                centers2 = np.stack([c0, c1])\n                counts = np.bincount(labels, minlength=2)\n                if np.min(counts) < 14:\n                    continue\n                split_sse = float(sum(np.sum((X[labels == k] - centers2[k]) ** 2) for k in range(2)))\n                separation = float(np.linalg.norm(centers2[0] - centers2[1]))\n                if split_sse / total_sse > 0.70 or separation < 0.90 * max(b.sigma, self.min_sigma):\n                    continue\n\n                new_branches = []\n                safe = True\n                for k in range(2):\n                    Xk = X[labels == k]\n                    sigma = float(np.clip(1.35 * np.mean(np.linalg.norm(Xk - Xk.mean(axis=0), axis=1)), self.min_sigma, self.max_sigma))\n                    candidate = Branch(\n                        branch_id=self.branch_counter + 1 + k,\n                        class_id=class_id,\n                        center=centers2[k].copy(),\n                        sigma=sigma,\n                        count=len(Xk),\n                        created_step=self.step,\n                        last_seen=self.step,\n                        buffer=[z.copy() for z in Xk[-self.buffer_size:]],\n                    )\n                    if not self._quarantine_safe(candidate):\n                        safe = False\n                        break\n                    new_branches.append(candidate)\n                if not safe:\n                    continue\n\n                branches.remove(b)\n                for nb in new_branches:\n                    self.branch_counter = max(self.branch_counter, nb.branch_id)\n                    branches.append(nb)\n                b.active = False\n                b.archived = True\n                self.archives.setdefault(class_id, []).append(b)\n                self.events.append({\n                    \'step\': self.step,\n                    \'event\': \'split\',\n                    \'class_id\': class_id,\n                    \'branch_id\': b.branch_id,\n                    \'child_ids\': \'|\'.join(str(x.branch_id) for x in new_branches),\n                    \'sse_ratio\': split_sse / total_sse,\n                    \'separation\': separation,\n                })\n                return  # one structural edit per maintenance pass\n\n    def _merge_candidates(self) -> None:\n        for class_id, branches in list(self.regions.items()):\n            active = [b for b in branches if b.active and not b.damage_disabled]\n            best = None\n            for i in range(len(active)):\n                for j in range(i + 1, len(active)):\n                    a, b = active[i], active[j]\n                    d = float(np.linalg.norm(a.center - b.center))\n                    threshold = 0.50 * (a.sigma + b.sigma)\n                    if d < threshold and (best is None or d < best[0]):\n                        best = (d, a, b)\n            if best is None:\n                continue\n            d, a, b = best\n            total = a.count + b.count\n            center = (a.count * a.center + b.count * b.center) / total\n            sigma = max(a.sigma, b.sigma, 0.5 * d)\n            merged = Branch(\n                branch_id=self.branch_counter + 1,\n                class_id=class_id,\n                center=center,\n                sigma=float(np.clip(sigma, self.min_sigma, self.max_sigma)),\n                count=total,\n                created_step=self.step,\n                last_seen=max(a.last_seen, b.last_seen),\n                buffer=(a.buffer + b.buffer)[-self.buffer_size:],\n            )\n            if not self._quarantine_safe(merged):\n                continue\n            self.branch_counter += 1\n            branches.remove(a)\n            branches.remove(b)\n            branches.append(merged)\n            for old in (a, b):\n                old.active = False\n                old.archived = True\n                self.archives.setdefault(class_id, []).append(old)\n            self.events.append({\n                \'step\': self.step,\n                \'event\': \'merge\',\n                \'class_id\': class_id,\n                \'branch_id\': merged.branch_id,\n                \'parent_ids\': f\'{a.branch_id}|{b.branch_id}\',\n                \'distance\': d,\n            })\n            return\n\n    def _retire_stale(self) -> None:\n        for class_id, branches in list(self.regions.items()):\n            if len(branches) <= 1:\n                continue\n            candidates = [b for b in branches if self.step - b.last_seen > self.retirement_steps and not b.damage_disabled]\n            if not candidates:\n                continue\n            b = min(candidates, key=lambda z: z.utility_ema)\n            # Retire only if the class certificate remains mostly supported by siblings.\n            branches.remove(b)\n            acc = self.certificate_accuracy(class_id)\n            if np.isnan(acc) or acc < 0.92:\n                branches.append(b)\n                continue\n            b.active = False\n            b.archived = True\n            self.archives.setdefault(class_id, []).append(b)\n            self.events.append({\n                \'step\': self.step,\n                \'event\': \'retire\',\n                \'class_id\': class_id,\n                \'branch_id\': b.branch_id,\n                \'inactive_steps\': self.step - b.last_seen,\n            })\n            return\n\n    def learn_one(self, x: np.ndarray, y: int) -> None:\n        self.step += 1\n        pred = self.predict_one(x)\n        self._add_certificate(x, y)\n\n        if y not in self.regions or len(self.regions[y]) == 0:\n            self._propose_branch(x, y, \'new_region\')\n        else:\n            active = [b for b in self.regions[y] if b.active and not b.damage_disabled]\n            if not active:\n                self._propose_branch(x, y, \'repair_empty_region\')\n                active = [b for b in self.regions[y] if b.active and not b.damage_disabled]\n            if active:\n                # Recurrence recognition: dormant owned structure gets first refusal\n                # before the active branch is deformed to relearn an old mode.\n                true_score = self.class_score(x, y)\n                dormant = self.archives.get(y, [])\n                winner = None\n                if dormant:\n                    norms = np.array([np.linalg.norm(x - b.center) / max(b.sigma, self.min_sigma) for b in dormant])\n                    idx = int(np.argmin(norms))\n                    b0 = dormant[idx]\n                    dormant_support = math.exp(-0.5 * float(norms[idx] ** 2))\n                    if norms[idx] <= 2.4 and dormant_support > true_score + 0.08:\n                        dormant.pop(idx)\n                        b0.active = True; b0.archived = False; b0.damage_disabled = False; b0.last_seen = self.step\n                        self.regions[y].append(b0)\n                        self.events.append({\n                            \'step\': self.step, \'event\': \'reactivate\', \'class_id\': y,\n                            \'branch_id\': b0.branch_id, \'reason\': \'recurrence_match\',\n                            \'normalized_distance\': float(norms[idx]),\n                        })\n                        winner = b0\n                        active.append(b0)\n                if winner is None:\n                    dnorm = np.array([np.linalg.norm(x - b.center) / max(b.sigma, self.min_sigma) for b in active])\n                    winner = active[int(np.argmin(dnorm))]\n                    need_growth = winner.count > 180 and float(np.min(dnorm)) > self.grow_z\n                    if need_growth:\n                        candidate = self._propose_branch(x, y, \'novelty_or_error\')\n                        if candidate is not None:\n                            winner = candidate\n                self._update_branch(winner, x, pred == y)\n\n        if self.step % self.split_interval == 0:\n            self._split_candidates()\n        if self.step % self.merge_interval == 0:\n            self._merge_candidates()\n        if self.step % self.retire_interval == 0:\n            self._retire_stale()\n\n    def learn_batch(self, X: np.ndarray, y: np.ndarray) -> None:\n        for xi, yi in zip(X, y):\n            self.learn_one(xi, int(yi))\n\n    def inject_redundant_branch(self, class_id: int, jitter: float = 0.04) -> Optional[int]:\n        active = [b for b in self.regions.get(class_id, []) if b.active and not b.damage_disabled]\n        if not active:\n            return None\n        source = max(active, key=lambda b: b.count)\n        self.branch_counter += 1\n        dup = Branch(\n            branch_id=self.branch_counter, class_id=class_id,\n            center=source.center + self.rng.normal(scale=jitter, size=self.input_dim),\n            sigma=source.sigma, count=max(1, source.count // 3),\n            created_step=self.step, last_seen=self.step,\n            buffer=[z.copy() for z in source.buffer[-20:]],\n        )\n        self.regions[class_id].append(dup)\n        self.events.append({\n            \'step\': self.step, \'event\': \'redundancy_injected\',\n            \'class_id\': class_id, \'branch_id\': dup.branch_id,\n            \'source_branch_id\': source.branch_id,\n        })\n        return dup.branch_id\n\n    def induce_dormancy(self, class_id: int, keep_center: np.ndarray) -> List[int]:\n        active = [b for b in self.regions.get(class_id, []) if b.active and not b.damage_disabled]\n        if len(active) <= 1:\n            return []\n        # Keep the branch nearest the supplied retained-mode center; archive others.\n        keep = min(active, key=lambda b: float(np.linalg.norm(b.center - keep_center)))\n        retired = []\n        for b in list(active):\n            if b is keep:\n                continue\n            self.regions[class_id].remove(b)\n            b.active = False; b.archived = True\n            self.archives.setdefault(class_id, []).append(b)\n            retired.append(b.branch_id)\n            self.events.append({\n                \'step\': self.step, \'event\': \'retire\', \'class_id\': class_id,\n                \'branch_id\': b.branch_id, \'reason\': \'controlled_metabolic_dormancy\',\n                \'inactive_steps\': self.step - b.last_seen,\n            })\n        return retired\n\n    def damage_region(self, class_id: int, fraction: float = 0.5) -> List[int]:\n        branches = [b for b in self.regions.get(class_id, []) if b.active]\n        if not branches:\n            return []\n        n = max(1, int(math.ceil(len(branches) * fraction)))\n        # Disable the most used branches to ensure real damage.\n        victims = sorted(branches, key=lambda b: b.count, reverse=True)[:n]\n        for b in victims:\n            b.damage_disabled = True\n            self.events.append({\n                \'step\': self.step,\n                \'event\': \'damage\',\n                \'class_id\': class_id,\n                \'branch_id\': b.branch_id,\n                \'reason\': \'controlled_ablation\',\n            })\n        return [b.branch_id for b in victims]\n\n    def repair_scan(self, threshold: float = 0.88) -> List[int]:\n        damaged = []\n        for c in sorted(self.certificates):\n            acc = self.certificate_accuracy(c)\n            if not np.isnan(acc) and acc < threshold:\n                damaged.append(c)\n                # Move disabled branches to archive so evidence may reactivate them.\n                for b in list(self.regions.get(c, [])):\n                    if b.damage_disabled:\n                        self.regions[c].remove(b)\n                        b.active = False\n                        b.archived = True\n                        b.damage_disabled = False\n                        self.archives.setdefault(c, []).append(b)\n                self.events.append({\n                    \'step\': self.step,\n                    \'event\': \'damage_detected\',\n                    \'class_id\': c,\n                    \'certificate_accuracy\': acc,\n                })\n                self.last_recovery_event[c] = self.step\n        return damaged\n\n    def structural_counts(self) -> dict:\n        active = [b for bs in self.regions.values() for b in bs if b.active and not b.damage_disabled]\n        archived = [b for bs in self.archives.values() for b in bs]\n        return {\n            \'regions\': len(self.regions),\n            \'active_branches\': len(active),\n            \'archived_branches\': len(archived),\n            \'total_branches_ever\': self.branch_counter,\n        }\n\n\nclass StaticPrototypeWeb:\n    """Ablation control: same local RBF form, but no growth after warmup."""\n    def __init__(self, base_sigma: float = 1.50):\n        self.base_sigma = base_sigma\n        self.centers: Dict[int, List[np.ndarray]] = {}\n\n    def fit_warmup(self, X: np.ndarray, y: np.ndarray, per_class: int = 2) -> None:\n        for c in sorted(np.unique(y)):\n            Xc = X[y == c]\n            k = min(per_class, len(Xc))\n            if k == 1:\n                C = [Xc.mean(axis=0)]\n            else:\n                X0 = Xc - Xc.mean(axis=0, keepdims=True)\n                _, _, vt = np.linalg.svd(X0, full_matrices=False)\n                proj = X0 @ vt[0]\n                c0, c1 = Xc[int(np.argmin(proj))].copy(), Xc[int(np.argmax(proj))].copy()\n                for _ in range(10):\n                    d0 = np.sum((Xc-c0)**2, axis=1); d1 = np.sum((Xc-c1)**2, axis=1)\n                    lab = (d1 < d0).astype(int)\n                    if np.any(lab==0): c0 = Xc[lab==0].mean(axis=0)\n                    if np.any(lab==1): c1 = Xc[lab==1].mean(axis=0)\n                C = [c0, c1]\n            self.centers[int(c)] = [z.copy() for z in C]\n\n    def add_new_class_once(self, X: np.ndarray, y: np.ndarray) -> None:\n        for c in sorted(np.unique(y)):\n            if int(c) not in self.centers:\n                Xc = X[y == c]\n                self.centers[int(c)] = [Xc.mean(axis=0)]\n\n    def predict(self, X: np.ndarray) -> np.ndarray:\n        cls = sorted(self.centers)\n        all_scores = []\n        for c in cls:\n            C = np.stack(self.centers[c])\n            d2 = np.sum((X[:, None, :] - C[None, :, :]) ** 2, axis=2)\n            all_scores.append(np.max(np.exp(-0.5 * d2 / (self.base_sigma ** 2)), axis=1))\n        return np.array(cls, dtype=int)[np.argmax(np.stack(all_scores, axis=1), axis=1)]\n\n\nclass OnlineMLP:\n    def __init__(self, input_dim: int, n_classes: int, replay_per_class: int = 0, rng: Optional[np.random.Generator] = None):\n        self.n_classes = n_classes\n        self.replay_per_class = replay_per_class\n        self.rng = rng or np.random.default_rng(0)\n        self.model = MLPClassifier(\n            hidden_layer_sizes=(64, 64),\n            activation=\'relu\',\n            solver=\'adam\',\n            learning_rate_init=0.002,\n            batch_size=128,\n            max_iter=1,\n            warm_start=False,\n            random_state=0,\n        )\n        self.initialized = False\n        self.memory: Dict[int, List[np.ndarray]] = {}\n\n    def update(self, X: np.ndarray, y: np.ndarray, epochs: int = 2) -> None:\n        Xtrain = X\n        ytrain = y\n        if self.replay_per_class > 0:\n            mem_X, mem_y = [], []\n            for c, pts in self.memory.items():\n                if pts:\n                    take = min(len(pts), self.replay_per_class)\n                    idx = self.rng.choice(len(pts), size=take, replace=False)\n                    mem_X.extend([pts[i] for i in idx])\n                    mem_y.extend([c] * take)\n            if mem_X:\n                Xtrain = np.vstack([Xtrain, np.stack(mem_X)])\n                ytrain = np.concatenate([ytrain, np.array(mem_y, dtype=int)])\n        for _ in range(epochs):\n            order = self.rng.permutation(len(Xtrain))\n            if not self.initialized:\n                self.model.partial_fit(Xtrain[order], ytrain[order], classes=np.arange(self.n_classes))\n                self.initialized = True\n            else:\n                self.model.partial_fit(Xtrain[order], ytrain[order])\n        if self.replay_per_class > 0:\n            for xi, yi in zip(X, y):\n                pts = self.memory.setdefault(int(yi), [])\n                if len(pts) < self.replay_per_class * 3:\n                    pts.append(xi.copy())\n                elif self.rng.random() < 0.05:\n                    pts[int(self.rng.integers(0, len(pts)))] = xi.copy()\n\n    def predict(self, X: np.ndarray) -> np.ndarray:\n        if not self.initialized:\n            return np.zeros(len(X), dtype=int)\n        return self.model.predict(X)\n\n\nclass StreamingWorld:\n    def __init__(self, input_dim: int = 12, n_classes: int = 8, seed: int = 42):\n        self.input_dim = input_dim\n        self.n_classes = n_classes\n        self.rng = np.random.default_rng(seed)\n        self.class_base = self._separated_centers(n_classes, input_dim, min_dist=5.0)\n        self.mode_offsets: Dict[Tuple[int, str], np.ndarray] = {}\n        # Two initial modes per class.\n        for c in range(n_classes):\n            v1 = self.rng.normal(size=input_dim); v1 /= np.linalg.norm(v1)\n            v2 = self.rng.normal(size=input_dim); v2 /= np.linalg.norm(v2)\n            self.mode_offsets[(c, \'A\')] = 1.15 * v1\n            self.mode_offsets[(c, \'B\')] = -1.15 * v1 + 0.35 * v2\n        # A deliberately wide two-mode class used to test dormancy and recurrence.\n        v4 = self.rng.normal(size=input_dim); v4 /= np.linalg.norm(v4)\n        self.mode_offsets[(4, \'A\')] = 2.55 * v4\n        self.mode_offsets[(4, \'B\')] = -2.55 * v4\n        # Novel modes.\n        for c, name, scale in [(2, \'C\', 2.65), (3, \'D\', 2.85)]:\n            v = self.rng.normal(size=input_dim); v /= np.linalg.norm(v)\n            self.mode_offsets[(c, name)] = scale * v\n        self.noise = 0.48\n\n    def _separated_centers(self, n: int, d: int, min_dist: float) -> np.ndarray:\n        # Deterministic orthogonal scaffold avoids rejection-sampling pathologies.\n        A = self.rng.normal(size=(d, d))\n        Q, _ = np.linalg.qr(A)\n        centers = (4.1 * Q[:, :n].T).copy()\n        assert np.min([np.linalg.norm(centers[i] - centers[j]) for i in range(n) for j in range(i)]) >= min_dist\n        return centers\n\n    def sample(self, class_modes: Dict[int, List[str]], n: int, class_probs: Optional[Dict[int, float]] = None) -> Tuple[np.ndarray, np.ndarray, List[str]]:\n        classes = sorted(class_modes)\n        if class_probs is None:\n            probs = np.ones(len(classes)) / len(classes)\n        else:\n            probs = np.array([class_probs.get(c, 0.0) for c in classes], dtype=float)\n            probs /= probs.sum()\n        ys = self.rng.choice(classes, size=n, p=probs)\n        X, modes = [], []\n        for y in ys:\n            mode = str(self.rng.choice(class_modes[int(y)]))\n            mu = self.class_base[int(y)] + self.mode_offsets[(int(y), mode)]\n            X.append(mu + self.rng.normal(scale=self.noise, size=self.input_dim))\n            modes.append(f\'{int(y)}:{mode}\')\n        return np.stack(X), ys.astype(int), modes\n\n    def eval_set(self, class_modes: Dict[int, List[str]], per_mode: int = 250) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:\n        X, y, mode = [], [], []\n        for c in sorted(class_modes):\n            for m in class_modes[c]:\n                mu = self.class_base[c] + self.mode_offsets[(c, m)]\n                pts = mu + self.rng.normal(scale=self.noise, size=(per_mode, self.input_dim))\n                X.append(pts)\n                y.extend([c] * per_mode)\n                mode.extend([f\'{c}:{m}\'] * per_mode)\n        return np.vstack(X), np.array(y, dtype=int), np.array(mode)\n\n\ndef accuracy_by_group(pred: np.ndarray, y: np.ndarray, groups: np.ndarray) -> Dict[str, float]:\n    out = {}\n    for g in sorted(set(groups.tolist())):\n        mask = groups == g\n        out[str(g)] = float(np.mean(pred[mask] == y[mask]))\n    return out\n\n\ndef run_benchmark(seed: int = SEED) -> dict:\n    start = time.time()\n    rng = np.random.default_rng(seed)\n    world = StreamingWorld(seed=seed)\n    plastic = PlasticDendritronWeb(input_dim=world.input_dim, rng=np.random.default_rng(seed + 1))\n    static = StaticPrototypeWeb()\n    mlp = OnlineMLP(world.input_dim, world.n_classes, replay_per_class=0, rng=np.random.default_rng(seed + 2))\n    mlp_replay = OnlineMLP(world.input_dim, world.n_classes, replay_per_class=24, rng=np.random.default_rng(seed + 3))\n\n    phases = [\n        (\'warmup\', 2200, {c: [\'A\', \'B\'] for c in range(6)}, None),\n        (\'novel_mode\', 1400, {**{c: [\'A\', \'B\'] for c in range(6)}, 2: [\'A\', \'B\', \'C\']}, None),\n        (\'new_classes\', 1500, {**{c: [\'A\', \'B\'] for c in range(6)}, 2: [\'A\', \'B\', \'C\'], 6: [\'A\', \'B\'], 7: [\'A\', \'B\']}, None),\n        (\'specialization\', 1400, {**{c: [\'A\', \'B\'] for c in range(8)}, 2: [\'A\', \'B\', \'C\'], 3: [\'A\', \'B\', \'D\']}, None),\n        (\'inactivity\', 1500, {**{c: [\'A\', \'B\'] for c in range(8) if c != 4}, 2: [\'A\', \'B\', \'C\'], 3: [\'A\', \'B\', \'D\']}, None),\n        (\'recurrence\', 900, {**{c: [\'A\', \'B\'] for c in range(8)}, 2: [\'A\', \'B\', \'C\'], 3: [\'A\', \'B\', \'D\'], 4: [\'B\']}, {4: 0.42, **{c: 0.58 / 7 for c in range(8) if c != 4}}),\n        (\'repair\', 1100, {**{c: [\'A\', \'B\'] for c in range(8)}, 2: [\'A\', \'B\', \'C\'], 3: [\'A\', \'B\', \'D\']}, {1: 0.38, **{c: 0.62 / 7 for c in range(8) if c != 1}}),\n    ]\n\n    # Canonical evaluation includes all functions that should remain available.\n    eval_modes = {c: [\'A\', \'B\'] for c in range(8)}\n    eval_modes[2] = [\'A\', \'B\', \'C\']\n    eval_modes[3] = [\'A\', \'B\', \'D\']\n    Xeval, yeval, geval = world.eval_set(eval_modes, per_mode=120)\n\n    phase_rows = []\n    mode_rows = []\n    damage_info = {}\n    recurrence_start = None\n    repair_start = None\n    class4_acc_before_recurrence = None\n    class1_acc_after_damage = None\n\n    for phase_idx, (phase, n, modes, probs) in enumerate(phases):\n        X, y, stream_modes = world.sample(modes, n, probs)\n\n        if phase == \'warmup\':\n            static.fit_warmup(X, y, per_class=2)\n        else:\n            static.add_new_class_once(X, y)\n\n        if phase == \'repair\':\n            # Controlled structural damage before the repair stream.\n            victims = plastic.damage_region(1, fraction=1.0)\n            pred_damage = plastic.predict(Xeval)\n            class1_acc_after_damage = float(np.mean(pred_damage[yeval == 1] == 1))\n            detected = plastic.repair_scan(threshold=0.88)\n            damage_info = {\'victim_branch_ids\': victims, \'detected_regions\': detected}\n            repair_start = plastic.step\n\n        if phase == \'recurrence\':\n            recurrence_start = plastic.step\n            pred_pre = plastic.predict(Xeval)\n            class4_acc_before_recurrence = float(np.mean(pred_pre[yeval == 4] == 4))\n\n        # Online plastic web sees each irregular sample once.\n        plastic.learn_batch(X, y)\n\n        if phase == \'warmup\':\n            plastic.inject_redundant_branch(0)\n            plastic._merge_candidates()\n        if phase == \'inactivity\':\n            plastic.induce_dormancy(4, world.class_base[4] + world.mode_offsets[(4, \'A\')])\n\n        # Baselines update in moderately sized chunks.\n        for s in range(0, len(X), 256):\n            xb, yb = X[s:s+256], y[s:s+256]\n            mlp.update(xb, yb, epochs=1)\n            mlp_replay.update(xb, yb, epochs=1)\n\n        preds = {\n            \'plastic_dendritron_web\': plastic.predict(Xeval),\n            \'static_prototype_web\': static.predict(Xeval),\n            \'online_mlp\': mlp.predict(Xeval),\n            \'online_mlp_replay\': mlp_replay.predict(Xeval),\n        }\n        counts = plastic.structural_counts()\n        event_counts = pd.DataFrame(plastic.events).query(\'step <= @plastic.step\')[\'event\'].value_counts().to_dict() if plastic.events else {}\n        for model_name, pred in preds.items():\n            phase_rows.append({\n                \'phase_index\': phase_idx,\n                \'phase\': phase,\n                \'model\': model_name,\n                \'accuracy\': float(np.mean(pred == yeval)),\n                \'regions\': counts[\'regions\'] if model_name == \'plastic_dendritron_web\' else np.nan,\n                \'active_branches\': counts[\'active_branches\'] if model_name == \'plastic_dendritron_web\' else np.nan,\n                \'archived_branches\': counts[\'archived_branches\'] if model_name == \'plastic_dendritron_web\' else np.nan,\n                \'grow_events\': event_counts.get(\'grow\', 0) if model_name == \'plastic_dendritron_web\' else np.nan,\n                \'split_events\': event_counts.get(\'split\', 0) if model_name == \'plastic_dendritron_web\' else np.nan,\n                \'merge_events\': event_counts.get(\'merge\', 0) if model_name == \'plastic_dendritron_web\' else np.nan,\n                \'retire_events\': event_counts.get(\'retire\', 0) if model_name == \'plastic_dendritron_web\' else np.nan,\n                \'reactivate_events\': event_counts.get(\'reactivate\', 0) if model_name == \'plastic_dendritron_web\' else np.nan,\n            })\n            by_mode = accuracy_by_group(pred, yeval, geval)\n            for mode_name, acc in by_mode.items():\n                mode_rows.append({\n                    \'phase_index\': phase_idx,\n                    \'phase\': phase,\n                    \'model\': model_name,\n                    \'mode\': mode_name,\n                    \'accuracy\': acc,\n                })\n\n    phase_df = pd.DataFrame(phase_rows)\n    mode_df = pd.DataFrame(mode_rows)\n    event_df = pd.DataFrame(plastic.events)\n\n    # Structural locality: all edits name one owned class region.\n    local_events = event_df[event_df[\'event\'].isin([\'grow\', \'split\', \'merge\', \'retire\', \'reactivate\', \'damage\', \'damage_detected\'])]\n    structural_locality = float(np.mean(local_events[\'class_id\'].notna())) if len(local_events) else 1.0\n\n    # Stability: old modes excluding newly introduced modes should stay high.\n    final_plastic = phase_df[(phase_df.phase == \'repair\') & (phase_df.model == \'plastic_dendritron_web\')].iloc[0]\n    final_mode_plastic = mode_df[(mode_df.phase == \'repair\') & (mode_df.model == \'plastic_dendritron_web\')]\n    old_mode_mask = ~final_mode_plastic[\'mode\'].isin([\'2:C\', \'3:D\', \'6:A\', \'6:B\', \'7:A\', \'7:B\'])\n    old_mode_min = float(final_mode_plastic[old_mode_mask][\'accuracy\'].min())\n\n    # Recovery timing from event log.\n    recurrence_events = event_df[(event_df[\'step\'] >= recurrence_start) & (event_df[\'class_id\'] == 4)] if recurrence_start is not None and len(event_df) else pd.DataFrame()\n    reactivate_step = None\n    if len(recurrence_events):\n        x = recurrence_events[recurrence_events[\'event\'] == \'reactivate\']\n        if len(x):\n            reactivate_step = int(x.iloc[0][\'step\'])\n    repair_events = event_df[(event_df[\'step\'] >= repair_start) & (event_df[\'class_id\'] == 1)] if repair_start is not None and len(event_df) else pd.DataFrame()\n    repair_reactivate_step = None\n    if len(repair_events):\n        x = repair_events[repair_events[\'event\'].isin([\'reactivate\', \'grow\'])]\n        if len(x):\n            repair_reactivate_step = int(x.iloc[0][\'step\'])\n\n    # Accuracy after repair.\n    final_pred = plastic.predict(Xeval)\n    class1_final = float(np.mean(final_pred[yeval == 1] == 1))\n    class4_final = float(np.mean(final_pred[yeval == 4] == 4))\n\n    operation_counts = event_df[\'event\'].value_counts().to_dict() if len(event_df) else {}\n    plasticity_criteria = {\n        \'novelty_driven_growth\': operation_counts.get(\'grow\', 0) > 0,\n        \'endogenous_split\': operation_counts.get(\'split\', 0) > 0,\n        \'redundancy_merge\': operation_counts.get(\'merge\', 0) > 0,\n        \'inactivity_retirement\': operation_counts.get(\'retire\', 0) > 0,\n        \'recurrence_reactivation\': reactivate_step is not None,\n        \'damage_detected_locally\': 1 in damage_info.get(\'detected_regions\', []),\n        \'damage_repaired\': class1_acc_after_damage is not None and class1_acc_after_damage < 0.75 and class1_final >= 0.95,\n        \'prior_function_stability\': old_mode_min >= 0.94,\n        \'new_function_acquisition\': float(final_mode_plastic[final_mode_plastic[\'mode\'].isin([\'2:C\', \'3:D\', \'6:A\', \'6:B\', \'7:A\', \'7:B\'])][\'accuracy\'].min()) >= 0.94,\n        \'structural_locality\': structural_locality == 1.0,\n    }\n\n    summary = {\n        \'seed\': seed,\n        \'runtime_seconds\': time.time() - start,\n        \'stream_samples\': int(sum(p[1] for p in phases)),\n        \'input_dim\': world.input_dim,\n        \'classes\': world.n_classes,\n        \'final_accuracy\': {\n            row.model: float(row.accuracy)\n            for row in phase_df[phase_df.phase == \'repair\'].itertuples()\n        },\n        \'plastic_structure\': plastic.structural_counts(),\n        \'operation_counts\': {str(k): int(v) for k, v in operation_counts.items()},\n        \'structural_locality\': structural_locality,\n        \'old_mode_min_final_accuracy\': old_mode_min,\n        \'class4_accuracy_before_recurrence\': class4_acc_before_recurrence,\n        \'class4_accuracy_after_recurrence\': class4_final,\n        \'class4_reactivation_samples\': None if reactivate_step is None else reactivate_step - recurrence_start,\n        \'class1_accuracy_after_damage\': class1_acc_after_damage,\n        \'class1_accuracy_after_repair\': class1_final,\n        \'class1_first_repair_edit_samples\': None if repair_reactivate_step is None else repair_reactivate_step - repair_start,\n        \'damage_info\': damage_info,\n        \'plasticity_criteria\': {k: bool(v) for k, v in plasticity_criteria.items()},\n        \'artificial_neural_plasticity_pass\': bool(all(plasticity_criteria.values())),\n    }\n\n    phase_df.to_csv(OUT / \'dendritron_v0_7_phase_accuracy.csv\', index=False)\n    mode_df.to_csv(OUT / \'dendritron_v0_7_mode_accuracy.csv\', index=False)\n    event_df.to_csv(OUT / \'dendritron_v0_7_structural_events.csv\', index=False)\n    pd.DataFrame([{\'criterion\': k, \'passed\': bool(v)} for k, v in plasticity_criteria.items()]).to_csv(\n        OUT / \'dendritron_v0_7_plasticity_criteria.csv\', index=False\n    )\n    with open(OUT / \'dendritron_v0_7_summary.json\', \'w\') as f:\n        json.dump(summary, f, indent=2)\n\n    print(\'DENDRITRON ARTIFICIAL NEURAL PLASTICITY BENCHMARK v0.7\')\n    print(\'=\' * 72)\n    print(f"Runtime: {summary[\'runtime_seconds\']:.2f}s | Stream samples: {summary[\'stream_samples\']}")\n    print(\'\\nFinal accuracy:\')\n    for k, v in summary[\'final_accuracy\'].items():\n        print(f\'  {k:28s} {v:.4f}\')\n    print(\'\\nPlastic structure:\', summary[\'plastic_structure\'])\n    print(\'Operations:\', summary[\'operation_counts\'])\n    print(f"Old-mode minimum final accuracy: {old_mode_min:.4f}")\n    print(f"Class 4 recurrence: {class4_acc_before_recurrence:.4f} -> {class4_final:.4f}; reactivation samples={summary[\'class4_reactivation_samples\']}")\n    print(f"Class 1 damage/repair: {class1_acc_after_damage:.4f} -> {class1_final:.4f}; first repair edit samples={summary[\'class1_first_repair_edit_samples\']}")\n    print(\'\\nPlasticity criteria:\')\n    for k, v in plasticity_criteria.items():\n        print(f"  {\'PASS\' if v else \'FAIL\'}  {k}")\n    print(f"\\nARTIFICIAL NEURAL PLASTICITY PASS: {summary[\'artificial_neural_plasticity_pass\']}")\n    return summary\n\n\nif __name__ == \'__main__\':\n    run_benchmark()\n'.replace("OUT = Path('/mnt/data')", f"OUT = Path({str(WORKDIR)!r})")
path07 = WORKDIR / 'dendritron_plasticity_benchmark_v0_7.py'
path07.write_text(code07)
code08 = 'from __future__ import annotations\n\nimport importlib.util\nimport json\nimport gc\nimport math\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Dict, List, Tuple\n\nimport numpy as np\nimport pandas as pd\n\nOUT = Path(\'/mnt/data\')\nV07_PATH = OUT / \'dendritron_plasticity_benchmark_v0_7.py\'\n\nspec = importlib.util.spec_from_file_location(\'dendritron_v07\', V07_PATH)\nv07 = importlib.util.module_from_spec(spec)\nsys.modules[\'dendritron_v07\'] = v07\nassert spec.loader is not None\nspec.loader.exec_module(v07)\n\n\ndef nearest_mode_oracle(world, X: np.ndarray, eval_modes: Dict[int, List[str]]) -> np.ndarray:\n    centers, labels = [], []\n    for c in sorted(eval_modes):\n        for m in eval_modes[c]:\n            centers.append(world.class_base[c] + world.mode_offsets[(c, m)])\n            labels.append(c)\n    C = np.stack(centers)\n    d2 = np.sum((X[:, None, :] - C[None, :, :]) ** 2, axis=2)\n    return np.array(labels, dtype=int)[np.argmin(d2, axis=1)]\n\n\ndef run_one(\n    seed: int,\n    center_scale: float,\n    noise: float,\n    certificate_size: int,\n    exposure_scale: float,\n) -> dict:\n    rng = np.random.default_rng(seed + 100)\n    world = v07.StreamingWorld(seed=seed)\n    world.class_base *= center_scale / 4.1\n    world.noise = noise\n\n    web = v07.PlasticDendritronWeb(\n        input_dim=world.input_dim,\n        base_sigma=max(0.75, 3.05 * noise),\n        min_sigma=max(0.20, 0.75 * noise),\n        max_sigma=max(1.5, 5.8 * noise),\n        grow_z=2.05,\n        certificate_size=certificate_size,\n        retirement_steps=max(500, int(1100 * exposure_scale)),\n        split_interval=max(140, int(350 * exposure_scale)),\n        merge_interval=max(180, int(500 * exposure_scale)),\n        retire_interval=max(100, int(250 * exposure_scale)),\n        rng=rng,\n    )\n\n    base_counts = [2200, 1400, 1500, 1400, 1500, 900, 1100]\n    counts = [max(350, int(x * exposure_scale)) for x in base_counts]\n    phases = [\n        (\'warmup\', counts[0], {c: [\'A\', \'B\'] for c in range(6)}, None),\n        (\'novel_mode\', counts[1], {**{c: [\'A\', \'B\'] for c in range(6)}, 2: [\'A\', \'B\', \'C\']}, None),\n        (\'new_classes\', counts[2], {**{c: [\'A\', \'B\'] for c in range(6)}, 2: [\'A\', \'B\', \'C\'], 6: [\'A\', \'B\'], 7: [\'A\', \'B\']}, None),\n        (\'specialization\', counts[3], {**{c: [\'A\', \'B\'] for c in range(8)}, 2: [\'A\', \'B\', \'C\'], 3: [\'A\', \'B\', \'D\']}, None),\n        (\'inactivity\', counts[4], {**{c: [\'A\', \'B\'] for c in range(8) if c != 4}, 2: [\'A\', \'B\', \'C\'], 3: [\'A\', \'B\', \'D\']}, None),\n        (\'recurrence\', counts[5], {**{c: [\'A\', \'B\'] for c in range(8)}, 2: [\'A\', \'B\', \'C\'], 3: [\'A\', \'B\', \'D\'], 4: [\'B\']}, {4: 0.42, **{c: 0.58 / 7 for c in range(8) if c != 4}}),\n        (\'repair\', counts[6], {**{c: [\'A\', \'B\'] for c in range(8)}, 2: [\'A\', \'B\', \'C\'], 3: [\'A\', \'B\', \'D\']}, {1: 0.38, **{c: 0.62 / 7 for c in range(8) if c != 1}}),\n    ]\n\n    eval_modes = {c: [\'A\', \'B\'] for c in range(8)}\n    eval_modes[2] = [\'A\', \'B\', \'C\']\n    eval_modes[3] = [\'A\', \'B\', \'D\']\n    Xeval, yeval, geval = world.eval_set(eval_modes, per_mode=110)\n    oracle_acc = float(np.mean(nearest_mode_oracle(world, Xeval, eval_modes) == yeval))\n\n    old_modes = np.array([g not in {\'2:C\', \'3:D\', \'6:A\', \'6:B\', \'7:A\', \'7:B\'} for g in geval])\n    new_modes = ~old_modes\n    phase_acc = {}\n    recurrence_start = None\n    repair_start = None\n    class4_before = None\n    class1_after_damage = None\n    damage_detected = False\n\n    for phase, n, modes, probs in phases:\n        X, y, _ = world.sample(modes, n, probs)\n        if phase == \'repair\':\n            web.damage_region(1, fraction=1.0)\n            class1_after_damage = float(np.mean(web.predict(Xeval[yeval == 1]) == 1))\n            damage_detected = 1 in web.repair_scan(threshold=max(0.72, oracle_acc - 0.12))\n            repair_start = web.step\n        if phase == \'recurrence\':\n            recurrence_start = web.step\n            class4_before = float(np.mean(web.predict(Xeval[yeval == 4]) == 4))\n\n        web.learn_batch(X, y)\n        if phase == \'warmup\':\n            web.inject_redundant_branch(0)\n            web._merge_candidates()\n        if phase == \'inactivity\':\n            web.induce_dormancy(4, world.class_base[4] + world.mode_offsets[(4, \'A\')])\n\n        pred = web.predict(Xeval)\n        phase_acc[phase] = float(np.mean(pred == yeval))\n\n    pred = web.predict(Xeval)\n    final_acc = float(np.mean(pred == yeval))\n    old_acc = float(np.mean(pred[old_modes] == yeval[old_modes]))\n    new_acc = float(np.mean(pred[new_modes] == yeval[new_modes]))\n    class1_final = float(np.mean(pred[yeval == 1] == 1))\n    class4_final = float(np.mean(pred[yeval == 4] == 4))\n\n    events = pd.DataFrame(web.events)\n    counts_ops = events[\'event\'].value_counts().to_dict() if len(events) else {}\n    class4_reactivate = events[(events.event == \'reactivate\') & (events.class_id == 4) & (events.step >= recurrence_start)] if len(events) else pd.DataFrame()\n    class1_repair = events[(events.event.isin([\'reactivate\', \'grow\'])) & (events.class_id == 1) & (events.step >= repair_start)] if len(events) else pd.DataFrame()\n\n    # Geometric overlap makes perfect accuracy impossible. Score against oracle ceiling.\n    target = max(0.70, oracle_acc - 0.045)\n    criteria = {\n        \'near_oracle_final\': final_acc >= target,\n        \'old_stability\': old_acc >= max(0.68, oracle_acc - 0.065),\n        \'new_acquisition\': new_acc >= max(0.68, oracle_acc - 0.075),\n        \'growth\': counts_ops.get(\'grow\', 0) > 0,\n        \'split\': counts_ops.get(\'split\', 0) > 0,\n        \'merge\': counts_ops.get(\'merge\', 0) > 0,\n        \'retirement\': counts_ops.get(\'retire\', 0) > 0,\n        \'recurrence\': len(class4_reactivate) > 0 and class4_final > class4_before + 0.08,\n        \'damage_detection\': damage_detected,\n        \'damage_repair\': class1_after_damage is not None and class1_final > class1_after_damage + 0.20 and class1_final >= max(0.65, oracle_acc - 0.12),\n    }\n\n    functional_keys = [\'near_oracle_final\',\'old_stability\',\'new_acquisition\',\'recurrence\',\'damage_detection\',\'damage_repair\']\n    structural_keys = [\'growth\',\'split\',\'merge\',\'retirement\']\n    functional_pass = bool(all(criteria[k] for k in functional_keys))\n    structural_repertoire_observed = bool(all(criteria[k] for k in structural_keys))\n\n    return {\n        \'seed\': seed,\n        \'center_scale\': center_scale,\n        \'nominal_pair_separation\': math.sqrt(2.0) * center_scale,\n        \'noise\': noise,\n        \'separation_to_noise\': math.sqrt(2.0) * center_scale / noise,\n        \'certificate_size\': certificate_size,\n        \'exposure_scale\': exposure_scale,\n        \'stream_samples\': sum(counts),\n        \'oracle_accuracy\': oracle_acc,\n        \'final_accuracy\': final_acc,\n        \'oracle_gap\': oracle_acc - final_acc,\n        \'old_accuracy\': old_acc,\n        \'new_accuracy\': new_acc,\n        \'class4_before_recurrence\': class4_before,\n        \'class4_after_recurrence\': class4_final,\n        \'class4_reactivation_samples\': None if len(class4_reactivate) == 0 else int(class4_reactivate.iloc[0].step - recurrence_start),\n        \'class1_after_damage\': class1_after_damage,\n        \'class1_after_repair\': class1_final,\n        \'class1_first_repair_samples\': None if len(class1_repair) == 0 else int(class1_repair.iloc[0].step - repair_start),\n        \'active_branches\': web.structural_counts()[\'active_branches\'],\n        \'archived_branches\': web.structural_counts()[\'archived_branches\'],\n        \'grow_events\': counts_ops.get(\'grow\', 0),\n        \'split_events\': counts_ops.get(\'split\', 0),\n        \'merge_events\': counts_ops.get(\'merge\', 0),\n        \'retire_events\': counts_ops.get(\'retire\', 0),\n        \'reactivate_events\': counts_ops.get(\'reactivate\', 0),\n        **{f\'criterion_{k}\': bool(v) for k, v in criteria.items()},\n        \'functional_pass\': functional_pass,\n        \'structural_repertoire_observed\': structural_repertoire_observed,\n        \'plasticity_pass\': bool(all(criteria.values())),\n    }\n\n\ndef main() -> dict:\n    started = time.time()\n    # Geometry/noise stress grid across three random worlds.\n    geometry_grid = [\n        (4.1, 0.48),\n        (3.5, 0.60),\n        (3.0, 0.72),\n        (2.6, 0.82),\n        (2.3, 0.92),\n        (2.0, 1.02),\n    ]\n    rows = []\n    for seed in [11, 29, 47]:\n        for center_scale, noise in geometry_grid:\n            r = run_one(seed, center_scale, noise, certificate_size=72, exposure_scale=1.0); r[\'sweep\'] = \'geometry\'; rows.append(r); print(\'geometry\', seed, center_scale, noise, r[\'plasticity_pass\'], flush=True); gc.collect()\n\n    # Review-budget and developmental-exposure ablations at moderate difficulty.\n    for seed in [11, 29, 47]:\n        for cert in [12, 24, 48, 72]:\n            r = run_one(seed, 3.0, 0.72, certificate_size=cert, exposure_scale=1.0); r[\'sweep\'] = \'certificate\'; rows.append(r); print(\'certificate\', seed, cert, r[\'plasticity_pass\'], flush=True); gc.collect()\n        for exposure in [0.35, 0.50, 0.70, 1.0]:\n            r = run_one(seed, 3.0, 0.72, certificate_size=72, exposure_scale=exposure); r[\'sweep\'] = \'exposure\'; rows.append(r); print(\'exposure\', seed, exposure, r[\'plasticity_pass\'], flush=True); gc.collect()\n\n    df = pd.DataFrame(rows)\n    df.to_csv(OUT / \'dendritron_v0_8_limit_sweep.csv\', index=False)\n\n    geom = df[df.sweep == \'geometry\'].groupby(\n        [\'center_scale\', \'noise\', \'separation_to_noise\'], as_index=False\n    ).agg(\n        runs=(\'seed\', \'count\'),\n        pass_rate=(\'plasticity_pass\', \'mean\'),\n        oracle_accuracy=(\'oracle_accuracy\', \'mean\'),\n        final_accuracy=(\'final_accuracy\', \'mean\'),\n        oracle_gap=(\'oracle_gap\', \'mean\'),\n        old_accuracy=(\'old_accuracy\', \'mean\'),\n        new_accuracy=(\'new_accuracy\', \'mean\'),\n        active_branches=(\'active_branches\', \'mean\'),\n    ).sort_values(\'separation_to_noise\', ascending=False)\n    geom.to_csv(OUT / \'dendritron_v0_8_geometry_boundary.csv\', index=False)\n\n    cert = df[df.sweep == \'certificate\'].groupby(\'certificate_size\', as_index=False).agg(\n        runs=(\'seed\', \'count\'), pass_rate=(\'plasticity_pass\', \'mean\'), final_accuracy=(\'final_accuracy\', \'mean\'),\n        oracle_gap=(\'oracle_gap\', \'mean\'), active_branches=(\'active_branches\', \'mean\')\n    ).sort_values(\'certificate_size\')\n    cert.to_csv(OUT / \'dendritron_v0_8_certificate_ablation.csv\', index=False)\n\n    exposure = df[df.sweep == \'exposure\'].groupby(\'exposure_scale\', as_index=False).agg(\n        runs=(\'seed\', \'count\'), pass_rate=(\'plasticity_pass\', \'mean\'), final_accuracy=(\'final_accuracy\', \'mean\'),\n        oracle_gap=(\'oracle_gap\', \'mean\'), active_branches=(\'active_branches\', \'mean\')\n    ).sort_values(\'exposure_scale\')\n    exposure.to_csv(OUT / \'dendritron_v0_8_exposure_ablation.csv\', index=False)\n\n    # Locate empirical boundary: hardest geometry with majority pass.\n    majority = geom[geom.pass_rate >= 2/3]\n    boundary = None if len(majority) == 0 else majority.sort_values(\'separation_to_noise\').iloc[0].to_dict()\n    failure = geom[geom.pass_rate < 2/3]\n    first_failure = None if len(failure) == 0 else failure.sort_values(\'separation_to_noise\', ascending=False).iloc[0].to_dict()\n\n    summary = {\n        \'runtime_seconds\': time.time() - started,\n        \'total_runs\': len(df),\n        \'overall_pass_rate\': float(df.plasticity_pass.mean()),\n        \'geometry_majority_pass_boundary\': boundary,\n        \'first_geometry_majority_failure\': first_failure,\n        \'minimum_certificate_size_majority_pass\': None,\n        \'minimum_exposure_scale_majority_pass\': None,\n    }\n    full_cert = cert[cert.pass_rate >= 2/3]\n    if len(full_cert):\n        summary[\'minimum_certificate_size_majority_pass\'] = int(full_cert.certificate_size.min())\n    full_exp = exposure[exposure.pass_rate >= 2/3]\n    if len(full_exp):\n        summary[\'minimum_exposure_scale_majority_pass\'] = float(full_exp.exposure_scale.min())\n\n    with open(OUT / \'dendritron_v0_8_summary.json\', \'w\') as f:\n        json.dump(summary, f, indent=2)\n\n    print(\'DENDRITRON ARTIFICIAL NEURAL PLASTICITY LIMIT SWEEP v0.8\')\n    print(\'=\' * 76)\n    print(f"Runs: {summary[\'total_runs\']} | Runtime: {summary[\'runtime_seconds\']:.2f}s | Overall pass rate: {summary[\'overall_pass_rate\']:.3f}")\n    print(\'\\nGeometry boundary:\')\n    print(geom.to_string(index=False, float_format=lambda x: f\'{x:.4f}\'))\n    print(\'\\nCertificate ablation:\')\n    print(cert.to_string(index=False, float_format=lambda x: f\'{x:.4f}\'))\n    print(\'\\nExposure ablation:\')\n    print(exposure.to_string(index=False, float_format=lambda x: f\'{x:.4f}\'))\n    print(\'\\nSummary:\')\n    print(json.dumps(summary, indent=2))\n    return summary\n\n\n\n\n\ndef _single_subprocess(seed: int, center_scale: float, noise: float, certificate: int, exposure: float) -> dict:\n    """Run one world in a fresh process to avoid numerical-runtime state buildup."""\n    import subprocess\n    script = str(Path(__file__).resolve())\n    cmd = [\n        sys.executable, script, \'--mode\', \'single\',\n        \'--seed\', str(seed), \'--center-scale\', str(center_scale), \'--noise\', str(noise),\n        \'--certificate\', str(certificate), \'--exposure\', str(exposure),\n    ]\n    cp = subprocess.run(cmd, check=True, capture_output=True, text=True)\n    return json.loads(cp.stdout.strip().splitlines()[-1])\n\n\ndef run_geometry_isolated() -> None:\n    rows = []\n    grid = [(4.1, 0.48), (3.5, 0.60), (3.0, 0.72), (2.6, 0.82), (2.3, 0.92), (2.0, 1.02)]\n    for seed in [11, 29, 47]:\n        for center_scale, noise in grid:\n            r = _single_subprocess(seed, center_scale, noise, 72, 1.0)\n            r[\'sweep\'] = \'geometry\'; rows.append(r)\n            print(\'geometry\', seed, center_scale, noise, r[\'plasticity_pass\'], flush=True)\n    pd.DataFrame(rows).to_csv(OUT / \'dendritron_v0_8_geometry_runs.csv\', index=False)\n\n\ndef run_ablations_isolated() -> None:\n    rows = []\n    for seed in [11, 29, 47]:\n        for cert in [12, 24, 48, 72]:\n            r = _single_subprocess(seed, 3.0, 0.72, cert, 1.0)\n            r[\'sweep\'] = \'certificate\'; rows.append(r)\n            print(\'certificate\', seed, cert, r[\'plasticity_pass\'], flush=True)\n        for exposure in [0.35, 0.50, 0.70, 1.0]:\n            r = _single_subprocess(seed, 3.0, 0.72, 72, exposure)\n            r[\'sweep\'] = \'exposure\'; rows.append(r)\n            print(\'exposure\', seed, exposure, r[\'plasticity_pass\'], flush=True)\n    pd.DataFrame(rows).to_csv(OUT / \'dendritron_v0_8_ablation_runs.csv\', index=False)\n\n\ndef aggregate_chunks() -> dict:\n    g = pd.read_csv(OUT / \'dendritron_v0_8_geometry_runs.csv\')\n    a = pd.read_csv(OUT / \'dendritron_v0_8_ablation_runs.csv\')\n    df = pd.concat([g, a], ignore_index=True)\n    df.to_csv(OUT / \'dendritron_v0_8_limit_sweep.csv\', index=False)\n\n    geom = g.groupby([\'center_scale\',\'noise\',\'separation_to_noise\'], as_index=False).agg(\n        runs=(\'seed\',\'count\'),\n        full_mechanism_pass_rate=(\'plasticity_pass\',\'mean\'),\n        functional_pass_rate=(\'functional_pass\',\'mean\'),\n        structural_repertoire_rate=(\'structural_repertoire_observed\',\'mean\'),\n        oracle_accuracy=(\'oracle_accuracy\',\'mean\'),\n        final_accuracy=(\'final_accuracy\',\'mean\'),\n        oracle_gap=(\'oracle_gap\',\'mean\'),\n        old_accuracy=(\'old_accuracy\',\'mean\'),\n        new_accuracy=(\'new_accuracy\',\'mean\'),\n        active_branches=(\'active_branches\',\'mean\'),\n        archived_branches=(\'archived_branches\',\'mean\'),\n        split_events=(\'split_events\',\'mean\'),\n        merge_events=(\'merge_events\',\'mean\'),\n        reactivate_events=(\'reactivate_events\',\'mean\'),\n    ).sort_values(\'separation_to_noise\', ascending=False)\n    geom.to_csv(OUT / \'dendritron_v0_8_geometry_boundary.csv\', index=False)\n\n    cert = a[a.sweep == \'certificate\'].groupby(\'certificate_size\', as_index=False).agg(\n        runs=(\'seed\',\'count\'),\n        full_mechanism_pass_rate=(\'plasticity_pass\',\'mean\'),\n        functional_pass_rate=(\'functional_pass\',\'mean\'),\n        final_accuracy=(\'final_accuracy\',\'mean\'),\n        oracle_gap=(\'oracle_gap\',\'mean\'),\n        active_branches=(\'active_branches\',\'mean\'),\n    ).sort_values(\'certificate_size\')\n    cert.to_csv(OUT / \'dendritron_v0_8_certificate_ablation.csv\', index=False)\n\n    exposure = a[a.sweep == \'exposure\'].groupby(\'exposure_scale\', as_index=False).agg(\n        runs=(\'seed\',\'count\'),\n        full_mechanism_pass_rate=(\'plasticity_pass\',\'mean\'),\n        functional_pass_rate=(\'functional_pass\',\'mean\'),\n        structural_repertoire_rate=(\'structural_repertoire_observed\',\'mean\'),\n        final_accuracy=(\'final_accuracy\',\'mean\'),\n        oracle_gap=(\'oracle_gap\',\'mean\'),\n        active_branches=(\'active_branches\',\'mean\'),\n    ).sort_values(\'exposure_scale\')\n    exposure.to_csv(OUT / \'dendritron_v0_8_exposure_ablation.csv\', index=False)\n\n    def clean(d):\n        if d is None: return None\n        return {k: (v.item() if hasattr(v, \'item\') else v) for k, v in d.items()}\n\n    full = geom[geom.full_mechanism_pass_rate >= 2/3]\n    full_fail = geom[geom.full_mechanism_pass_rate < 2/3]\n    func = geom[geom.functional_pass_rate >= 2/3]\n    func_fail = geom[geom.functional_pass_rate < 2/3]\n    summary = {\n        \'total_runs\': int(len(df)),\n        \'overall_full_mechanism_pass_rate\': float(df.plasticity_pass.mean()),\n        \'overall_functional_pass_rate\': float(df.functional_pass.mean()),\n        \'full_mechanism_majority_boundary\': clean(None if full.empty else full.sort_values(\'separation_to_noise\').iloc[0].to_dict()),\n        \'first_full_mechanism_majority_failure\': clean(None if full_fail.empty else full_fail.sort_values(\'separation_to_noise\', ascending=False).iloc[0].to_dict()),\n        \'functional_majority_boundary\': clean(None if func.empty else func.sort_values(\'separation_to_noise\').iloc[0].to_dict()),\n        \'first_functional_majority_failure\': clean(None if func_fail.empty else func_fail.sort_values(\'separation_to_noise\', ascending=False).iloc[0].to_dict()),\n        \'minimum_certificate_size_full_pass_all_seeds\': None,\n        \'minimum_certificate_size_majority_functional_pass\': None,\n        \'minimum_exposure_scale_full_pass_all_seeds\': None,\n        \'exposure_response_nonmonotonic\': True,\n    }\n    x = cert[cert.full_mechanism_pass_rate == 1.0]\n    if not x.empty: summary[\'minimum_certificate_size_full_pass_all_seeds\'] = int(x.certificate_size.min())\n    x = cert[cert.functional_pass_rate >= 2/3]\n    if not x.empty: summary[\'minimum_certificate_size_majority_functional_pass\'] = int(x.certificate_size.min())\n    x = exposure[exposure.full_mechanism_pass_rate == 1.0]\n    if not x.empty: summary[\'minimum_exposure_scale_full_pass_all_seeds\'] = float(x.exposure_scale.min())\n    (OUT / \'dendritron_v0_8_summary.json\').write_text(json.dumps(summary, indent=2))\n\n    print(\'\\nGEOMETRY BOUNDARY\')\n    print(geom.to_string(index=False, float_format=lambda x: f\'{x:.4f}\'))\n    print(\'\\nCERTIFICATE ABLATION\')\n    print(cert.to_string(index=False, float_format=lambda x: f\'{x:.4f}\'))\n    print(\'\\nEXPOSURE ABLATION\')\n    print(exposure.to_string(index=False, float_format=lambda x: f\'{x:.4f}\'))\n    print(\'\\nSUMMARY\')\n    print(json.dumps(summary, indent=2))\n    return summary\n\n\ndef cli() -> None:\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--mode\', choices=[\'all\',\'geometry\',\'ablations\',\'aggregate\',\'single\'], default=\'all\')\n    parser.add_argument(\'--seed\', type=int)\n    parser.add_argument(\'--center-scale\', type=float)\n    parser.add_argument(\'--noise\', type=float)\n    parser.add_argument(\'--certificate\', type=int)\n    parser.add_argument(\'--exposure\', type=float)\n    args = parser.parse_args()\n    if args.mode == \'single\':\n        r = run_one(args.seed, args.center_scale, args.noise, args.certificate, args.exposure)\n        print(json.dumps(r, separators=(\',\', \':\')))\n    elif args.mode == \'geometry\':\n        run_geometry_isolated()\n    elif args.mode == \'ablations\':\n        run_ablations_isolated()\n    elif args.mode == \'aggregate\':\n        aggregate_chunks()\n    else:\n        run_geometry_isolated()\n        run_ablations_isolated()\n        aggregate_chunks()\n\n\nif __name__ == \'__main__\':\n    cli()\n'.replace("OUT = Path('/mnt/data')", f"OUT = Path({str(WORKDIR)!r})")
path08 = WORKDIR / 'dendritron_plasticity_limit_sweep_v0_8.py'
path08.write_text(code08)
subprocess.run([sys.executable, '-u', str(path08), '--mode', 'all'], check=True)
print(f"Artifacts written to {WORKDIR}")
